In [2]:
import os
import glob
import cv2
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.autolayout': True
})

In [ ]:
def get_video_info(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps and fps > 0 else 0
    
    cap.release()
    
    return {
        'filename': os.path.basename(video_path),
        'fps': round(fps, 2),
        'frame_count': frame_count,
        'resolution': f"{width}x{height}",
        'width': width,
        'height': height,
        'duration_sec': round(duration, 2)
    }

def collect_dataset_metadata(labels_dict, dataset_name):
    records = []
    print(f"Start dataset: {dataset_name}...")
    
    for folder_path, category_name in labels_dict.items():
        if not os.path.exists(folder_path):
            print(f"not found: {folder_path}")
            continue

        is_real = category_name.startswith('Real')
        binary_label = 'Real' if is_real else 'Fake'
        
        video_files = []
        for ext in ('*.mp4', '*.avi', '*.mov'):
            video_files.extend(glob.glob(os.path.join(folder_path, ext)))
            video_files.extend(glob.glob(os.path.join(folder_path, "**", ext), recursive=True))
        
        video_files = list(set(video_files))
        print(f" Found {len(video_files)} videos in: {folder_path} [{category_name}]")
        
        for v_path in video_files:
            info = get_video_info(v_path)
            if info:
                info['dataset'] = dataset_name
                info['category'] = category_name
                info['label'] = binary_label
                records.append(info)
                
    return pd.DataFrame(records)

In [12]:
ff_labels = {
    'datasets/FaceForensics++_C23/original': 'Real',
    'datasets/FaceForensics++_C23/DeepFakeDetection': 'Fake (DeepFakeDetection)',
    'datasets/FaceForensics++_C23/Deepfakes': 'Fake (Deepfakes)',
    'datasets/FaceForensics++_C23/Face2Face': 'Fake (Face2Face)',
    'datasets/FaceForensics++_C23/FaceShifter': 'Fake (FaceShifter)',
    'datasets/FaceForensics++_C23/FaceSwap': 'Fake (FaceSwap)',
    'datasets/FaceForensics++_C23/NeuralTextures': 'Fake (NeuralTextures)',
}


celeb_labels = {
    'datasets/Celeb/Celeb-real': 'Real (Celeb)',
    'datasets/Celeb/YouTube-real': 'Real (YouTube)',
    'datasets/Celeb/Celeb-synthesis': 'Fake'
}

In [ ]:
CSV_CACHE = "datasets_eda_metadata.csv"

if os.path.exists(CSV_CACHE):
    print(f"Metadata is present '{CSV_CACHE}'.")
    df_all = pd.read_csv(CSV_CACHE)
else:
    df_ff = collect_dataset_metadata(ff_labels, 'FaceForensics++ (C23)')
    df_celeb = collect_dataset_metadata(celeb_labels, 'Celeb-DF v2')

    df_all = pd.concat([df_ff, df_celeb], ignore_index=True)

    if not df_all.empty:
        df_all.to_csv(CSV_CACHE, index=False)
        print(f"\n Metadata saved: '{CSV_CACHE}'\n")
    else:
        print("No videos found")

In [ ]:
summary_table = df_all.groupby(['dataset', 'label', 'category']).agg(
    video_count=('filename', 'count'),
    avg_duration_sec=('duration_sec', 'mean'),
    avg_frames=('frame_count', 'mean'),
    main_resolution=('resolution', lambda x: x.value_counts().index[0])
).reset_index()


plt.figure(figsize=(10, 6))
df_ff_only = df_all[df_all['dataset'] == 'FaceForensics++ (C23)'].copy()

ff_categories_order = [
    'Real',
    'Fake (Deepfakes)',
    'Fake (Face2Face)',
    'Fake (FaceSwap)',
    'Fake (NeuralTextures)',
    'Fake (FaceShifter)',
    'Fake (DeepFakeDetection)'
]
existing_ff_order = [cat for cat in ff_categories_order if cat in df_ff_only['category'].unique()]

fake_count = len(existing_ff_order) - 1
reds_gradient = sns.color_palette("Reds", n_colors=fake_count + 2)[2:]

ff_palette = {}
fake_idx = 0
for cat in existing_ff_order:
    if cat.startswith('Real'):
        ff_palette[cat] = '#2ca02c'  # green
    else:
        ff_palette[cat] = reds_gradient[fake_idx]
        fake_idx += 1

ax1 = sns.countplot(
    data=df_ff_only, 
    x='category', 
    order=existing_ff_order,
    palette=ff_palette
)
plt.title('FaceForensics++ (C23): Class Distribution', pad=14)
plt.xlabel('')
plt.ylabel('')
plt.xticks(rotation=25, ha='right')

# plt.savefig('fig1_ff_categories.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

plt.figure(figsize=(8, 6))
df_celeb_only = df_all[df_all['dataset'] == 'Celeb-DF v2'].copy()

celeb_categories_order = ['Real (Celeb)', 'Real (YouTube)', 'Fake']
existing_celeb_order = [cat for cat in celeb_categories_order if cat in df_celeb_only['category'].unique()]

celeb_palette = {
    'Real (Celeb)': '#2ca02c',
    'Real (YouTube)': '#52be80',
    'Fake': '#d62728'
}

ax2 = sns.countplot(
    data=df_celeb_only, 
    x='category', 
    order=existing_celeb_order,
    palette=celeb_palette
)
plt.title('Celeb-DF v2: Class Distribution', pad=14)
plt.xlabel('')
plt.ylabel('')


# plt.savefig('fig2_celeb_categories.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

plt.figure(figsize=(9, 5))
sns.histplot(
    data=df_all, 
    x='duration_sec', 
    hue='dataset', 
    element='step',
    stat='density',
    common_norm=False,
    kde=True,
    palette=['#1f77b4', '#ff7f0e']
)
plt.title('Video Length Distribution (s)', pad=12)
plt.xlabel('')
plt.ylabel('')

# plt.savefig('fig3_video_duration_kde.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

plt.figure(figsize=(8, 5))
binary_counts = df_all.groupby(['dataset', 'label']).size().reset_index(name='count')

ax4 = sns.barplot(
    data=binary_counts, 
    x='dataset', 
    y='count', 
    hue='label', 
    palette={'Real': '#2ca02c', 'Fake': '#d62728'},
    order = ['FaceForensics++ (C23)', 'Celeb-DF v2']
)
plt.title('Total Class Disbalance', pad=12)
plt.xlabel('')
plt.ylabel('')

# plt.savefig('fig4_binary_imbalance.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
import random

N_SAMPLES = 4

In [ ]:
def extract_random_frame(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return None

    start_frame = int(total_frames * 0.2)
    end_frame = max(start_frame + 1, int(total_frames * 0.8))
    target_frame = random.randint(start_frame, end_frame)

    cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
    ret, frame = cap.read()
    cap.release()

    if ret and frame is not None:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return None

def find_video_path(filename):
    for root, _, files in os.walk('datasets'):
        if filename in files:
            return os.path.join(root, filename)
    return None

def plot_samples_for_dataset(df_all, dataset_name, output_filename, n_samples=4):
    df_ds = df_all[df_all['dataset'] == dataset_name]
    
    if df_ds.empty:
        print(f"datset not found^'{dataset_name}'.")
        return

    df_real = df_ds[df_ds['label'] == 'Real']
    df_fake = df_ds[df_ds['label'] == 'Fake']

    sample_real = df_real.sample(n=min(n_samples, len(df_real)))
    sample_fake = df_fake.sample(n=min(n_samples, len(df_fake)))

    fig, axes = plt.subplots(nrows=2, ncols=n_samples, figsize=(3.5 * n_samples, 7))

    for col_idx, (_, row) in enumerate(sample_real.iterrows()):
        ax = axes[0][col_idx]
        v_path = find_video_path(row['filename'])
        frame = extract_random_frame(v_path) if v_path else None

        ax.imshow(frame)
        ax.set_title(f"Real", fontsize=10, color='darkgreen', fontweight='bold')
        ax.axis('off')

    for col_idx, (_, row) in enumerate(sample_fake.iterrows()):
        ax = axes[1][col_idx]
        v_path = find_video_path(row['filename'])
        frame = extract_random_frame(v_path) if v_path else None

        ax.imshow(frame)
        ax.set_title(f"Fake", fontsize=10, color='darkred', fontweight='bold')
        ax.axis('off')

    plt.tight_layout()
    # plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
df_metadata = pd.read_csv(CSV_CACHE)

plot_samples_for_dataset(
    df_all=df_metadata, 
    dataset_name='FaceForensics++ (C23)', 
    output_filename='fig5_ff_samples.png',
    n_samples=N_SAMPLES
)

plot_samples_for_dataset(
    df_all=df_metadata, 
    dataset_name='Celeb-DF v2', 
    output_filename='fig6_celeb_samples.png',
    n_samples=N_SAMPLES
)

In [ ]:
min_row = df_metadata.loc[df_metadata['duration_sec'].idxmin()]
print("=== Least duration ===")
print(f"File: {min_row['filename']}")
print(f"Dataset: {min_row['dataset']}")
print(f"Category: {min_row['category']}")
print(f"Duration: {min_row['duration_sec']} s. ({min_row['frame_count']} frames {min_row['fps']} FPS)\n")

print("=== MIN duration per dataset ===")
min_by_ds = df_metadata.groupby('dataset')['duration_sec'].min()
print(min_by_ds.to_string())

# print("\n=== 5 shortest video ===")
# shortest_5 = df_metadata.sort_values(by='duration_sec')[['filename', 'dataset', 'category', 'duration_sec', 'frame_count']].head(5)
# print(shortest_5.to_string(index=False))

In [ ]:
threshold_sec=5
print(f"=== Videos duration < {threshold_sec} s ===\n")

datasets = df_metadata['dataset'].unique()

for ds_name in datasets:
    df_metadata_ds = df_metadata[df_metadata['dataset'] == ds_name]
    total_count = len(df_metadata_ds)

    short_videos = df_metadata_ds[df_metadata_ds['duration_sec'] < threshold_sec]
    short_count = len(short_videos)
    short_percentage = (short_count / total_count) * 100 if total_count > 0 else 0

    print(f"Dataset: {ds_name}")
    print(f"-Total video: {total_count}")
    print(f"-Videos < {threshold_sec} s: {short_count}")
    print(f"- Short %: {short_percentage:.2f}%\n")

total_all = len(df_metadata)
short_all = len(df_metadata[df_metadata['duration_sec'] < threshold_sec])
total_percentage = (short_all / total_all) * 100 if total_all > 0 else 0

print("--------------------------------------------------")
print(f"Both datasets total:")
print(f"- Total videos: {total_all}")
print(f"- Videos < {threshold_sec} s: {short_all}")
print(f"- Total %: {total_percentage:.2f}%")
print("--------------------------------------------------\n")

print("=== Real vs Fake ===")
breakdown = df_metadata.groupby(['dataset', 'label']).agg(
    total_videos=('filename', 'count'),
    short_videos=('duration_sec', lambda x: (x < threshold_sec).sum()),
).reset_index()

breakdown['short_share_%'] = (breakdown['short_videos'] / breakdown['total_videos'] * 100).round(2)
print(breakdown.to_string(index=False))